In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use("Agg")
import numpy as np
import os

# PART 1 – PrunableLinear Layer

class PrunableLinear(nn.Module):
    """
    A drop-in replacement for nn.Linear that associates each weight
    with a learnable scalar gate in [0, 1].

    Forward pass:
        gates       = sigmoid(gate_scores)          # shape: (out, in)
        pruned_w    = weight * gates                # element-wise
        output      = pruned_w @ x.T + bias
    """

    def __init__(self, in_features: int, out_features: int):
        super().__init__()
        self.in_features  = in_features
        self.out_features = out_features

        self.weight      = nn.Parameter(torch.empty(out_features, in_features))
        self.bias        = nn.Parameter(torch.zeros(out_features))

        self.gate_scores = nn.Parameter(torch.zeros(out_features, in_features))

        nn.init.kaiming_uniform_(self.weight, nonlinearity="relu")

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        gates         = torch.sigmoid(self.gate_scores)
        pruned_weights = self.weight * gates
        return F.linear(x, pruned_weights, self.bias)

    def get_gates(self) -> torch.Tensor:
        """Return current gate values (detached) for analysis."""
        return torch.sigmoid(self.gate_scores).detach()

    def sparsity_penalty(self) -> torch.Tensor:
        """L1 norm of gate values (sum of all gate activations)."""
        return torch.sigmoid(self.gate_scores).sum()

    def extra_repr(self) -> str:
        return f"in={self.in_features}, out={self.out_features}"

# Network Definition

class SelfPruningNet(nn.Module):
    """
    Feed-forward network for CIFAR-10 classification.
    CIFAR-10: 32×32 RGB images → 10 classes
    Input dim: 3072 (3 × 32 × 32)
    """

    def __init__(self):
        super().__init__()
        self.fc1 = PrunableLinear(3072, 512)
        self.fc2 = PrunableLinear(512, 256)
        self.fc3 = PrunableLinear(256, 128)
        self.fc4 = PrunableLinear(128, 10)

        self.bn1 = nn.BatchNorm1d(512)
        self.bn2 = nn.BatchNorm1d(256)
        self.bn3 = nn.BatchNorm1d(128)

        self.dropout = nn.Dropout(0.3)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x.view(x.size(0), -1)
        x = F.relu(self.bn1(self.fc1(x)))
        x = self.dropout(x)
        x = F.relu(self.bn2(self.fc2(x)))
        x = self.dropout(x)
        x = F.relu(self.bn3(self.fc3(x)))
        x = self.fc4(x)
        return x

    def sparsity_loss(self) -> torch.Tensor:
        """Sum of L1 gate penalties across all PrunableLinear layers."""
        total = sum(
            layer.sparsity_penalty()
            for layer in self.modules()
            if isinstance(layer, PrunableLinear)
        )
        return total

    def all_gates(self) -> torch.Tensor:
        """Concatenated gate values from all PrunableLinear layers."""
        gates = [
            layer.get_gates().flatten()
            for layer in self.modules()
            if isinstance(layer, PrunableLinear)
        ]
        return torch.cat(gates)


# PART 3 – Training & Evaluation

def get_data_loaders(batch_size: int = 128):
    """Download CIFAR-10 and return train/test loaders."""
    mean = (0.4914, 0.4822, 0.4465)
    std  = (0.247,  0.243,  0.261)

    train_transform = transforms.Compose([
        transforms.RandomHorizontalFlip(),
        transforms.RandomCrop(32, padding=4),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])
    test_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])

    train_set = datasets.CIFAR10(root="./data", train=True,  download=True, transform=train_transform)
    test_set  = datasets.CIFAR10(root="./data", train=False, download=True, transform=test_transform)

    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True,  num_workers=2, pin_memory=True)
    test_loader  = DataLoader(test_set,  batch_size=256,        shuffle=False, num_workers=2, pin_memory=True)
    return train_loader, test_loader


def train_one_epoch(model, loader, optimizer, device, lam: float):
    """Single epoch: compute total_loss = CE + λ * L1_gates."""
    model.train()
    total_loss = 0.0
    correct    = 0
    total      = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        logits = model(images)

        ce_loss       = F.cross_entropy(logits, labels)
        sparse_loss   = model.sparsity_loss()
        loss          = ce_loss + lam * sparse_loss

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += images.size(0)

    return total_loss / total, 100.0 * correct / total


@torch.no_grad()
def evaluate(model, loader, device):
    """Return test accuracy (%)."""
    model.eval()
    correct = total = 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        logits  = model(images)
        correct += (logits.argmax(1) == labels).sum().item()
        total   += images.size(0)
    return 100.0 * correct / total


def compute_sparsity(model, threshold: float = 1e-2) -> float:
    """% of weights whose gate value < threshold."""
    gates = model.all_gates().cpu().numpy()
    return 100.0 * (gates < threshold).sum() / len(gates)


def run_experiment(lam: float, epochs: int, device, train_loader, test_loader):
    """Train a fresh model for a given λ and return (test_acc, sparsity, model)."""
    print(f"\n{'='*55}")
    print(f"  Running experiment: λ = {lam}")
    print(f"{'='*55}")

    model     = SelfPruningNet().to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    for epoch in range(1, epochs + 1):
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, device, lam)
        scheduler.step()
        if epoch % 5 == 0 or epoch == 1:
            test_acc = evaluate(model, test_loader, device)
            sparsity = compute_sparsity(model)
            print(f"  Epoch {epoch:3d}/{epochs} | Loss: {train_loss:.4f} | "
                  f"Train: {train_acc:.1f}% | Test: {test_acc:.1f}% | Sparse: {sparsity:.1f}%")

    final_test_acc = evaluate(model, test_loader, device)
    final_sparsity = compute_sparsity(model)
    print(f"\n  ► Final Test Accuracy : {final_test_acc:.2f}%")
    print(f"  ► Sparsity Level      : {final_sparsity:.2f}%")
    return final_test_acc, final_sparsity, model


def plot_gate_distribution(model, lam: float, save_path: str):
    """
    Histogram of gate values.
    A well-pruned model shows a large spike near 0 and a
    secondary cluster away from 0 (active weights).
    """
    gates = model.all_gates().cpu().numpy()

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(gates, bins=100, color="#2a9d8f", edgecolor="none", alpha=0.85)
    ax.set_xlabel("Gate Value", fontsize=12)
    ax.set_ylabel("Count", fontsize=12)
    ax.set_title(f"Gate Value Distribution  (λ = {lam})", fontsize=14, fontweight="bold")
    ax.axvline(x=0.01, color="#e76f51", linestyle="--", linewidth=1.5, label="Prune threshold (0.01)")
    ax.legend(fontsize=10)
    fig.tight_layout()
    fig.savefig(save_path, dpi=150)
    plt.close(fig)
    print(f"  Plot saved → {save_path}")


def save_results_table(results: list, save_path: str):
    """Save a simple text summary of results."""
    lines = [
        "Lambda | Test Accuracy | Sparsity Level (%)",
        "-------|---------------|-------------------",
    ]
    for lam, acc, sparse in results:
        lines.append(f"{lam:<7}| {acc:.2f}%         | {sparse:.2f}%")
    with open(save_path, "w") as f:
        f.write("\n".join(lines))
    print(f"  Results table saved → {save_path}")

# Main

def main():
    EPOCHS       = 30
    BATCH_SIZE   = 128
    LAMBDAS      = [1e-5, 1e-4, 5e-4]
    PRUNE_THRESH = 1e-2
    RESULTS_DIR  = "./results"
    os.makedirs(RESULTS_DIR, exist_ok=True)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"\nDevice: {device}")

    train_loader, test_loader = get_data_loaders(BATCH_SIZE)

    results     = []
    best_acc    = -1
    best_model  = None
    best_lam    = None

    for lam in LAMBDAS:
        acc, sparsity, model = run_experiment(lam, EPOCHS, device, train_loader, test_loader)
        results.append((lam, acc, sparsity))

        plot_path = os.path.join(RESULTS_DIR, f"gate_dist_lam_{lam}.png")
        plot_gate_distribution(model, lam, plot_path)

        if acc > best_acc:
            best_acc   = acc
            best_model = model
            best_lam   = lam

    #Summary
    print("\n\n" + "="*55)
    print("  FINAL RESULTS SUMMARY")
    print("="*55)
    print(f"{'Lambda':<12} {'Test Acc':>12} {'Sparsity':>14}")
    print("-"*40)
    for lam, acc, sparse in results:
        print(f"{lam:<12} {acc:>11.2f}%  {sparse:>12.2f}%")

    # Save table
    save_results_table(results, os.path.join(RESULTS_DIR, "results_table.txt"))

    # Save best model
    torch.save(best_model.state_dict(), os.path.join(RESULTS_DIR, "best_model.pt"))
    print(f"\n  Best model (λ={best_lam}, acc={best_acc:.2f}%) saved.")

    # Combined gate distribution for best model (used in report)
    plot_gate_distribution(best_model, best_lam,
                           os.path.join(RESULTS_DIR, "gate_dist_best.png"))

    print("\nDone. All outputs written to ./results/")


if __name__ == "__main__":
    main()



Device: cpu


100%|██████████| 170M/170M [00:02<00:00, 78.3MB/s]



  Running experiment: λ = 1e-05


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


  Epoch   1/30 | Loss: 10.4853 | Train: 31.7% | Test: 40.5% | Sparse: 0.0%
  Epoch   5/30 | Loss: 10.1729 | Train: 42.2% | Test: 48.6% | Sparse: 0.0%
  Epoch  10/30 | Loss: 10.0956 | Train: 45.2% | Test: 50.7% | Sparse: 0.0%
  Epoch  15/30 | Loss: 10.0328 | Train: 47.8% | Test: 52.9% | Sparse: 0.0%
  Epoch  20/30 | Loss: 9.9826 | Train: 49.5% | Test: 55.3% | Sparse: 0.0%
  Epoch  25/30 | Loss: 9.9337 | Train: 51.4% | Test: 56.4% | Sparse: 0.0%
  Epoch  30/30 | Loss: 9.9135 | Train: 52.2% | Test: 57.0% | Sparse: 0.0%

  ► Final Test Accuracy : 56.96%
  ► Sparsity Level      : 0.00%
  Plot saved → ./results/gate_dist_lam_1e-05.png

  Running experiment: λ = 0.0001
  Epoch   1/30 | Loss: 83.0453 | Train: 31.4% | Test: 40.4% | Sparse: 0.0%
  Epoch   5/30 | Loss: 77.8862 | Train: 42.3% | Test: 48.8% | Sparse: 0.0%
  Epoch  10/30 | Loss: 77.7747 | Train: 45.4% | Test: 51.1% | Sparse: 0.0%
  Epoch  15/30 | Loss: 77.7100 | Train: 47.8% | Test: 53.7% | Sparse: 0.0%
  Epoch  20/30 | Loss: 77.659